In [4]:
!pip install ddgs sentence-transformers transformers accelerate bitsandbytes 


In [5]:
import re
import torch
from ddgs import DDGS
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM


In [11]:
class ClaimExtractor:

    def extract_claims(self, article):

        sentences = re.split(r'[.!?]+', article)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 25]

        claims = sentences[:3]

        return claims

class QueryGenerator:

    def generate_queries(self, claims):

        queries = []

        for claim in claims:

            # ✅ FULL CLAIM (VERY IMPORTANT)
            queries.append(claim)

            # ✅ keyword version
            words = re.findall(r'\b[A-Za-z0-9]+\b', claim)
            keywords = [w for w in words if len(w) > 3]

            keyword_query = " ".join(keywords[:8])

            if keyword_query.strip():
                queries.append(keyword_query)

        return list(set(queries))[:5]
        
from ddgs import DDGS
from urllib.parse import urlparse

class KnowledgeRetriever:

    def __init__(self):

        # ✅ Expanded credible domains
        self.credible_domains = {

            # High trust news
            "reuters.com":1.0,
            "apnews.com":1.0,
            "bbc.com":1.0,
            "bbc.co.uk":1.0,
            "wikipedia.org":1.0,
            "britannica.com":0.95,
            "encyclopedia.com":0.9,

            # Major media
            "nytimes.com":0.95,
            "theguardian.com":0.95,
            "washingtonpost.com":0.95,
            "wsj.com":0.95,
            "bloomberg.com":0.95,

            # Fact-checking ⭐
            "snopes.com":1.0,
            "factcheck.org":1.0,
            "politifact.com":1.0,

            # Science / health
            "nature.com":0.95,
            "science.org":0.95,
            "sciencedaily.com":0.9,
            "nih.gov":1.0,
            "who.int":1.0,
            "cdc.gov":1.0,

            # Educational / government
            ".edu":0.9,
            ".gov":1.0
        }

    # ✅ Extract domain cleanly
    def extract_domain(self, url):

        try:
            domain = urlparse(url).netloc.lower()
            domain = domain.replace("www.", "")
            return domain
        except:
            return ""

    # ✅ Assign credibility score
    def get_credibility(self, domain):

        for d, score in self.credible_domains.items():
            if d in domain:
                return score

        return 0.3  # default low

    # ✅ Detect source type (optional but useful)
    def get_source_type(self, domain):

        if ".gov" in domain or ".edu" in domain:
            return "HIGH_TRUST"

        if any(d in domain for d in ["reuters","bbc","apnews"]):
            return "NEWS"

        if any(d in domain for d in ["snopes","factcheck","politifact"]):
            return "FACT_CHECK"

        return "UNKNOWN"

    # ✅ MAIN SEARCH FUNCTION
    def search(self, query):

        evidence = []

        if query.strip() == "":
            return evidence

        try:
            with DDGS() as ddgs:

                results = ddgs.text(query, max_results=10)

                for r in results:

                    url = r.get("href", "")
                    title = r.get("title", "")
                    snippet = r.get("body", "")

                    domain = self.extract_domain(url)

                    credibility = self.get_credibility(domain)

                    # ✅ FILTER LOW QUALITY
                    if credibility < 0.6:
                        continue

                    evidence.append({
                        "title": title,
                        "snippet": snippet,
                        "url": url,
                        "domain": domain,
                        "credibility_score": credibility,
                        "source_type": self.get_source_type(domain)
                    })

        except Exception as e:
            print("Search Error:", e)

        # ✅ REMOVE DUPLICATES
        unique = []
        seen = set()

        for e in evidence:
            if e["url"] not in seen:
                unique.append(e)
                seen.add(e["url"])

        return unique

class EvidenceRanker:

    def __init__(self):
        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')

    def rank(self, claim, evidence):

        if len(evidence) == 0:
            return []

        claim_emb = self.encoder.encode(claim, convert_to_tensor=True)

        snippets = [e["snippet"] for e in evidence]
        snippet_emb = self.encoder.encode(snippets, convert_to_tensor=True)

        similarities = util.cos_sim(claim_emb, snippet_emb)[0]

        for i, e in enumerate(evidence):

            semantic = float(similarities[i])
            credibility = e["credibility_score"]

            # ✅ improved weighting
            e["score"] = 0.6 * semantic + 0.4 * credibility

        ranked = sorted(evidence, key=lambda x: x["score"], reverse=True)

        return ranked[:5]
        

class ClaimVerifier:

    def __init__(self):

        model_name = "mistralai/Mistral-7B-Instruct-v0.2"

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto"
        )

    def verify(self, claim, evidence):

        # ---------- Handle empty evidence ----------
        if len(evidence) == 0:
            return "UNVERIFIED",0.4,"Insufficient evidence"

        # ---------- Prepare evidence ----------
        top_evidence = evidence[:3]
        evidence_text = "\n".join([e["snippet"] for e in top_evidence])

        # ---------- Improved Prompt ----------
        prompt = f"""
You are a fact-checking assistant.

Claim:
{claim}

Evidence:
{evidence_text}

Task:
- Check whether evidence supports or contradicts the claim
- Use only given evidence

Respond EXACTLY like this:

VERDICT: TRUE or FALSE
CONFIDENCE: number between 0 and 1
EXPLANATION: short explanation based on evidence
"""

        # ---------- Tokenize ----------
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        # ---------- Generate ----------
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.0,
                pad_token_id=self.tokenizer.eos_token_id
            )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        # ---------- Default values ----------
        verdict = "REAL"
        confidence = 0.5
        explanation = "Could not determine clearly."

        # ---------- Safe Parsing ----------
        for line in response.split("\n"):

            line_upper = line.upper()

            if "VERDICT" in line_upper:
                if "FALSE" in line_upper:
                    verdict = "FAKE"
                elif "TRUE" in line_upper:
                    verdict = "REAL"

            elif "CONFIDENCE" in line_upper:
                import re
                nums = re.findall(r"\d*\.?\d+", line)
                if nums:
                    confidence = float(nums[0])

            elif "EXPLANATION" in line_upper:
                parts = line.split(":", 1)
                if len(parts) > 1:
                    explanation = parts[1].strip()

        # ---------- Add Source-Based Explanation ----------
        high_cred_sources = [e for e in evidence if e["credibility_score"] >= 0.9]
        domains = [e["domain"] for e in top_evidence if "domain" in e]

        if verdict == "REAL":
            explanation += f" Supported by {len(high_cred_sources)} high-credibility sources."
        else:
            explanation += f" Contradicted by credible sources."

        if domains:
            explanation += f" Sources include: {', '.join(domains[:2])}."

        return verdict, confidence, explanation

class FakeNewsPipeline:

    def __init__(self):

        self.extractor = ClaimExtractor()
        self.query_gen = QueryGenerator()
        self.retriever = KnowledgeRetriever()
        self.ranker = EvidenceRanker()
        self.verifier = ClaimVerifier()

    def detect(self, article):

        claims = self.extractor.extract_claims(article)

        if not claims:
            return None

        results = []

        for claim in claims:

            queries = self.query_gen.generate_queries([claim])

            evidence = []

            for q in queries:
                evidence.extend(self.retriever.search(q))

            ranked = self.ranker.rank(claim, evidence)

            verdict, conf, exp = self.verifier.verify(claim, ranked)

            results.append({
                "verdict": verdict,
                "confidence": conf
            })

        # ✅ weighted decision
        fake_score = sum(r["confidence"] for r in results if r["verdict"] == "FAKE")
        real_score = sum(r["confidence"] for r in results if r["verdict"] == "REAL")

        if fake_score > real_score * 1.1:
            final = "FAKE"
        elif real_score > fake_score * 1.1:
            final = "REAL"
        else:
            final = "PARTIALLY FAKE"

        avg_conf = sum(r["confidence"] for r in results) / len(results)

        return {
            "verdict": final,
            "confidence": avg_conf
        }

In [13]:
import pandas as pd

liar_df = pd.read_csv(
    "/kaggle/input/datasets/doanquanvietnamca/liar-dataset/train.tsv",
    sep="\t",
    header=None
)

# rename important columns
liar_df.columns = [
    "id","label","statement","subject","speaker",
    "speaker_job","state","party","barely_true",
    "false","half_true","mostly_true","pants_fire","context"
]

# keep only needed columns
liar_df = liar_df[["statement","label"]]

liar_df.head()
liar_df["label"] = liar_df["label"].apply(
    lambda x: "FAKE" if x in ["false","pants-fire","barely-true"] else "REAL"
)


In [ ]:
pipeline = FakeNewsPipeline()

y_true=[]
y_pred=[]

fake_samples = liar_df[liar_df["label"]=="FAKE"].sample(50, random_state=42)
real_samples = liar_df[liar_df["label"]=="REAL"].sample(50, random_state=42)

data = pd.concat([fake_samples, real_samples])

for _,row in data.iterrows():

    claim=row["statement"]
    label=row["label"]

    result=pipeline.detect(claim)

    predicted=result["verdict"]

    y_true.append(label)
    y_pred.append(predicted)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [11]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report

accuracy=accuracy_score(y_true,y_pred)

precision=precision_score(y_true,y_pred,pos_label="REAL")

recall=recall_score(y_true,y_pred,pos_label="REAL")

f1=f1_score(y_true,y_pred,pos_label="REAL")

print("\n==============================")
print("MODEL EVALUATION")
print("==============================")

print("Accuracy:",round(accuracy,4))
print("Precision:",round(precision,2))
print("Recall:",round(recall,2))
print("F1 Score:",round(f1,2))

print("\nDetailed Report:\n")
print(classification_report(y_true,y_pred))


MODEL EVALUATION
Accuracy: 0.5725
Precision: 0.58
Recall: 0.51
F1 Score: 0.54

Detailed Report:

              precision    recall  f1-score   support

        FAKE       0.56      0.64      0.60       200
        REAL       0.58      0.51      0.54       200

    accuracy                           0.57       400
   macro avg       0.57      0.57      0.57       400
weighted avg       0.57      0.57      0.57       400

